In [ ]:
%pip install -q deep_translator 

In [ ]:
# Install deep_translator if not already installed
import subprocess
import sys

try:
    import deep_translator
except ImportError:
    print("Installing deep_translator...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "deep_translator", "-q"])
    print("deep_translator installed.")

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root
file_path = PROJECT_ROOT / "data" / "processed" / "lyrics_lang.csv"
# DATABRICKS PATH
# file_path = "/Volumes/songs_db/default/storage/lyrics_lang.csv"

df = pd.read_csv(file_path)

print(df.head())

   rank                            artist                    title    region  \
0     1        Mr Plata, El Americano 4KT           Las Muñequitas  Colombia   
1     2            ARIA VEGA, Ryan Castro  CHÉVERE (premium_remix)  Colombia   
2     3        Ryan Castro, Kapo, Gangsta                 LA VILLA  Colombia   
3     4                           Kris R.                    GANAS  Colombia   
4     5  W Sound, Beéle, Ovy On The Drums    La Plena - W Sound 05  Colombia   

              spotify_uri                                             lyrics  \
0  4nJJCRYru4QQakCiUA155f  (Dímelo, ¿me vas a dar lo que yo pido?)\nDame ...   
1  3CBEVPwR3kUXDoTx1lqFUQ  ARIA VEGA, Ryan Castro\nLa costeñita premium y...   
2  2ZyrAym0sRLwt4PhGotHuI  Kapo, Ryan Castro, Gangsta\nQué chimba, SOG\nT...   
3  4KE9Ne3hgh18B3Th4xcylg  Yeah, yeah\nYeah, yeah\n\nMi amor, culeemos co...   
4  6iOndD4OFo7GkaDypWQIou  O-O-Ovy On The Drums\nYou're the apple of my e...   

  language  
0       es  
1       es  

In [ ]:
import duckdb as dd
con = dd.connect()
lang_groups = con.execute("SELECT original_lang, COUNT(*) AS count FROM df GROUP BY original_lang order by original_lang").df()

display(lang_groups)

,language,count
0,ar,2
1,de,3
2,en,340
3,es,211
4,fr,3
5,gd,2
6,he,1
7,id,3
8,it,3
9,ja,4


In [ ]:
df_lyrics_lang = con.execute("""
            SELECT spotify_uri, left(lyrics, 100) AS lyrics, original_lang
            FROM df 
            where 1=1
            and original_lang is not null
            and original_lang not in ('en', 'unknown')
            order by original_lang
            """).df()

display(df_lyrics_lang)

,spotify_uri,lyrics,language
0,54PbBpquVfhfrwRwvjSXbI,لم نعد نتحدث، لم نعد نتحدث\nلم نعد نتحدث كما ك...,ar
1,54PbBpquVfhfrwRwvjSXbI,لم نعد نتحدث، لم نعد نتحدث\nلم نعد نتحدث كما ك...,ar
2,2gam98EZKrF9XuOkU13ApN,Lenk' ich dich ab?\nNein\nDachte ich mir\n\nWi...,de
3,5rb9QrpfcKFHM1EUbSIurX,Peace! Und das Zeichen für die Stadt mit A\nJa...,de
4,3B54sVLJ402zGa6Xm4YGNe,Es ist nicht gut genug für mich\nSeitdem ich m...,de
...,...,...,...
346,5TkQbhQm9BXONk2agDo4w9,還沒好好的感受 雪花綻放的氣候\n我們一起顫抖 會更明白什麼是溫柔\n還沒跟你牽著手 走過荒...,zh
347,2fc2vUA5H3mTAr9rQoBDKt,抓不住愛情的我、總是眼睜睜看她溜走\n世界上幸福的人到處有、為何不能算我一個\n為了愛孤軍奮...,zh
348,4OoExItZJ0jePoCZDbHx4t,你的回話凌亂著 在這個時刻\n我想起噴泉旁的白鴿 甜蜜散落了\n情緒莫名的拉扯 我還愛你呢\...,zh
349,73Ijz3vN9KP1W5qrfNeWMI,我知 妳老母都嫌我醜\n我知 妳老爸都看我不爽快\n我知 妳家的狗 咬我三次\n全家伙都賭爛...,zh


In [4]:
# Cell 4
from pathlib import Path
import re
import pandas as pd
import concurrent.futures as cf
from deep_translator import GoogleTranslator

LANGUAGE_CODE_MAP = {
    "zh-cn": "zh-CN",
    "zh_cn": "zh-CN",
    "zh-tw": "zh-TW",
    "zh_tw": "zh-TW",
    "jp": "ja",
    "kr": "ko",
    "latin": "auto",
    "other": "auto",
    "unknown": "auto",
}

CJK_PATTERN = re.compile(r"[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]")
TRANSLATE_TIMEOUT_SEC = 30
translator_cache = {}

def normalize_source_language(language_value):
    if pd.isna(language_value):
        return "auto"
    language = str(language_value).strip().lower()
    if not language:
        return "auto"
    return LANGUAGE_CODE_MAP.get(language, language)

def has_cjk(text):
    if pd.isna(text):
        return False
    return bool(CJK_PATTERN.search(str(text)))

def get_translator(source_language):
    if source_language not in translator_cache:
        translator_cache[source_language] = GoogleTranslator(source=source_language, target="en")
    return translator_cache[source_language]

def translate_once(text, source_language, timeout_sec=TRANSLATE_TIMEOUT_SEC):
    try:
        translator = get_translator(source_language)
        with cf.ThreadPoolExecutor(max_workers=1) as ex:
            fut = ex.submit(translator.translate, text)
            return fut.result(timeout=timeout_sec)
    except cf.TimeoutError:
        return pd.NA
    except Exception:
        return pd.NA

def translate_line_by_line(text, source_language):
    lines = str(text).splitlines()
    out_lines = []

    for line in lines:
        line_stripped = line.strip()
        if not line_stripped:
            out_lines.append("")
            continue

        translated = translate_once(line_stripped, source_language)
        if pd.isna(translated):
            translated = translate_once(line_stripped, "auto")
        if pd.isna(translated):
            translated = line_stripped

        out_lines.append(str(translated))

    return "\n".join(out_lines)

def translate_to_english(text, source_language="auto"):
    if pd.isna(text):
        return pd.NA

    original = str(text).strip()
    if not original:
        return pd.NA

    normalized_source = normalize_source_language(source_language)

    first_try = translate_once(original, normalized_source)
    if pd.isna(first_try) and normalized_source != "auto":
        first_try = translate_once(original, "auto")

    if pd.isna(first_try):
        first_try = translate_line_by_line(original, normalized_source)

    unchanged = (not pd.isna(first_try)) and (str(first_try).strip() == original)
    still_cjk = (not pd.isna(first_try)) and has_cjk(first_try)

    if unchanged or still_cjk:
        retry_auto = translate_once(original, "auto")
        if not pd.isna(retry_auto) and str(retry_auto).strip() != original and not has_cjk(retry_auto):
            return retry_auto

        retry_lines = translate_line_by_line(original, "auto")
        if retry_lines and retry_lines.strip():
            return retry_lines

    return first_try

In [ ]:
# Cell 
# Translate all non-English rows; keep English rows as null in lyrics_en
df_translated = df.copy()

if "lyrics" not in df_translated.columns or "original_lang" not in df_translated.columns:
    raise KeyError("Expected columns 'lyrics' and 'original_lang' in the input CSV")

output_path = PROJECT_ROOT / "data" / "processed" / "lyrics_trans.csv"
# DATABRICKS PATH
# output_path = Path("/Volumes/songs_db/default/storage/lyrics_trans.csv")
checkpoint_every = 1

# Resume support: continue from prior partial output if present
if output_path.exists():
    existing = pd.read_csv(output_path)
    if "lyrics_in_en" in existing.columns and len(existing) == len(df_translated):
        df_translated["lyrics_in_en"] = existing["lyrics_in_en"]
    else:
        df_translated["lyrics_in_en"] = pd.NA
else:
    df_translated["lyrics_in_en"] = pd.NA

non_english_mask = df_translated["original_lang"].fillna("").str.lower() != "en"
has_lyrics_mask = df_translated["lyrics"].notna()

existing_en = df_translated["lyrics_in_en"].fillna("").astype(str).str.strip()
original_lyrics = df_translated["lyrics"].fillna("").astype(str).str.strip()

still_cjk_mask = existing_en.str.contains(r"[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]", regex=True)
unchanged_mask = existing_en.eq(original_lyrics) & existing_en.ne("")

needs_translation_mask = non_english_mask & has_lyrics_mask & (
    df_translated["lyrics_in_en"].isna() | still_cjk_mask | unchanged_mask
)

pending_indices = df_translated.index[needs_translation_mask].tolist()
total_pending = len(pending_indices)
print(f"Pending translations: {total_pending:,}")

since_last_save = 0
processed = 0
failed_rows = []

for idx in pending_indices:
    uri = df_translated.at[idx, "spotify_uri"] if "spotify_uri" in df_translated.columns else idx
    print(f"Translating {processed + 1}/{total_pending} | uri={uri} | lang={df_translated.at[idx, 'original_lang']}")

    translated = translate_to_english(
        df_translated.at[idx, "lyrics"],
        source_language=df_translated.at[idx, "original_lang"],
    )

    if pd.isna(translated):
        failed_rows.append(uri)

    df_translated.at[idx, "lyrics_in_en"] = translated
    since_last_save += 1
    processed += 1

    if since_last_save >= checkpoint_every:
        df_translated[["artist", "title", "spotify_uri", "original_lang", "lyrics_in_en"]].to_csv(output_path, index=False)
        print(f"Progress: {processed}/{total_pending} (checkpoint saved)")
        since_last_save = 0

# For English rows, copy lyrics to lyrics_in_en (per plan Phase 3)
english_mask = df_translated["original_lang"].fillna("").str.lower() == "en"
df_translated.loc[english_mask, "lyrics_in_en"] = df_translated.loc[english_mask, "lyrics"]

# Final save (only output required columns)
df_translated[["artist", "title", "spotify_uri", "original_lang", "lyrics_in_en"]].to_csv(output_path, index=False)

print(f"Progress: {processed}/{total_pending} (final save)")
print(f"Wrote {len(df_translated):,} rows to {output_path}")
print(f"Failed rows (timeout/error): {len(failed_rows):,}")

if failed_rows:
    failed_path = output_path.with_name("lyrics_trans_failed_uris.csv")
    pd.DataFrame({"spotify_uri": failed_rows}).to_csv(failed_path, index=False)
    print(f"Wrote failed URI list to {failed_path}")

display(df_translated[["spotify_uri", "original_lang", "lyrics_in_en"]].head(10))

Pending translations: 129
Translating 1/129 | uri=0bQSWXtpau3VkNI1ZcXTQA | lang=zh
Progress: 1/129 (checkpoint saved)
Translating 2/129 | uri=0VTzUEuHYD8s7CgQ15cDPo | lang=zh
Progress: 2/129 (checkpoint saved)
Translating 3/129 | uri=76EfpqmO6JUL2TTR1SIGwz | lang=zh
Progress: 3/129 (checkpoint saved)
Translating 4/129 | uri=4UwZyC0wEV6XDEAr9kaQhi | lang=zh
Progress: 4/129 (checkpoint saved)
Translating 5/129 | uri=0cOMncRq4cmDLO4tPQnkBF | lang=zh
Progress: 5/129 (checkpoint saved)
Translating 6/129 | uri=1NoDTQhJsrd5rnpb6PQthK | lang=zh
Progress: 6/129 (checkpoint saved)
Translating 7/129 | uri=3wofqqt5YJWdhLI9IFaX39 | lang=zh
Progress: 7/129 (checkpoint saved)
Translating 8/129 | uri=4Zwn8WxD61diTwqv9hyTQA | lang=zh
Progress: 8/129 (checkpoint saved)
Translating 9/129 | uri=3xJK8ywApYVEYIDhymYMC6 | lang=zh
Progress: 9/129 (checkpoint saved)
Translating 10/129 | uri=7z2M7DsEjZjwXBkWG3zd21 | lang=zh
Progress: 10/129 (checkpoint saved)
Translating 11/129 | uri=59lBAMCis4C6NsPdUV35Vz | la